# Deep-Space Link Agent: full pipeline
**Matlab in Space Hackathon · Track 1: Deep-Space Communication & Signal Intelligence**

One notebook, from input data to results:

```
RadioML (2016 or 2018) ─► train modulation classifier ─► saved model
                                                       │
synthetic telemetry ─► deep-space channel ─► autonomous receiver agent ─► decoded telemetry + plots
                       (Doppler, drift, hop,     (find signal, remove Doppler,
                        dropout, noise)           identify modulation, demodulate,
                                                  re-acquire after losses)
```

**Before running:**
1. Put the dataset in the `data/` folder: `RML2016.10a_dict.pkl` (2016) or `GOLD_XYZ_OSC.0001_1024.hdf5` (2018, needs `pip install h5py`), and set `DATASET` below.
2. Open Jupyter from inside the `deepspace-link/` folder (the notebook uses the code in `src/`).
3. **Kernel → Restart & Run All.** Everything is written to `results/` and `models/`.

## 0. Settings: the only cell you might edit

In [ ]:
DATASET    = "2016"       # "2016" (RadioML 2016.10A) | "2018" (RadioML 2018.01A, needs h5py)
DATA_PATH  = None         # None = standard file name in data/
PER_PAIR   = 200          # 2018 only: examples per (modulation, SNR). 200 -> ~125k examples, ~1 GB RAM
SNR_MIN    = None         # 2018 only: e.g. -10 to skip the very noisy examples
MODEL      = "features"   # "features" (fast, no GPU) | "cnn" (needs PyTorch) | "both"
EPOCHS     = 15           # CNN only
ESN0_DB    = 8.0          # signal quality of the simulated pass (try 5–6 for harder)
N_FRAMES   = 300          # telemetry frames in the pass
RUN_SWEEP  = True         # frame success vs SNR (about 1 minute)
SEED       = 0

In [ ]:
# !pip install numpy scipy matplotlib pandas scikit-learn torch
import os, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch
from sklearn.metrics import confusion_matrix

sys.path.insert(0, os.getcwd())          # so "src" is importable from the notebook
from src.radioml import load_dataset, split, DEFAULT_PATH, DEFAULT_PATH_2018
from src.classifier import ModulationClassifier, train_features_model, HAVE_TORCH
from src.transmitter import transmit
from src.channel import deep_space_channel
from src.receiver import SignalAgent

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
SPS = 8
os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)
# ---------- progress tracking ----------
T_START = time.time()
def step(title):
    # banner at the start of each stage: wall-clock time and time since start
    print(f"\n{'=' * 72}\n[{time.strftime('%H:%M:%S')}] (+{time.time() - T_START:7.1f} s)  {title}\n{'=' * 72}")
def info(msg):
    print(f"   [{time.strftime('%H:%M:%S')}] {msg}")

step("STEP 0  Setup")
info(f"Python {sys.version.split()[0]} | numpy {np.__version__} | PyTorch: {'yes' if HAVE_TORCH else 'no'}")
info(f"settings: DATASET={DATASET}, MODEL={MODEL}, ESN0_DB={ESN0_DB}, N_FRAMES={N_FRAMES}, RUN_SWEEP={RUN_SWEEP}")
if MODEL in ("cnn", "both") and not HAVE_TORCH:
    info("PyTorch not installed, falling back to MODEL = 'features'  (pip install torch)")
    MODEL = "features"

## 1. Input data: RadioML
| | 2016.10A | 2018.01A |
|---|---|---|
| File | pickle, 641 MB | HDF5, 21.4 GB |
| Modulations | 11 | 24 |
| SNR | −20 … +18 dB | −20 … +30 dB |
| Samples per example | 128 | 1,024 |
| Loaded here | everything | a balanced subset (`PER_PAIR` per modulation and SNR), since the full file doesn't fit in RAM |

Each example is normalized to unit power, like the receiver's AGC.

In [ ]:
step(f"STEP 1  Load RadioML {DATASET}")
path = DATA_PATH or (DEFAULT_PATH_2018 if DATASET == "2018" else DEFAULT_PATH)
info(f"reading {path} ({os.path.getsize(path)/1e6:.0f} MB)..." if os.path.exists(path) else f"looking for {path}...")
t0 = time.time()
kw = dict(per_pair=PER_PAIR, snr_min=SNR_MIN, seed=SEED) if DATASET == "2018" else {}
X, y, snr, MODS, SNRS = load_dataset(DATASET, path, **kw)
L = X.shape[-1]                                  # 128 (2016) or 1024 (2018)
TAG = "" if DATASET == "2016" else "_2018"       # 2016 and 2018 outputs don't overwrite each other
info(f"loaded {len(y)} examples in {time.time()-t0:.1f} s | array {X.shape}, {X.nbytes/1e6:.0f} MB in memory")
info(f"{len(MODS)} modulations: {MODS}")
info(f"SNR {SNRS[0]}..{SNRS[-1]} dB")
idx_tr, idx_te = split(y, snr, seed=SEED)
info(f"split: train {len(idx_tr)} | test {len(idx_te)} (stratified by modulation and SNR)")
info("plotting one example per modulation...")

cols = 4 if len(MODS) <= 12 else 6
rows = int(np.ceil(len(MODS) / cols))
fig, ax = plt.subplots(rows, cols, figsize=(3.5 * cols, 2.6 * rows))
for a in ax.flat:
    a.axis("off")
for a, m in zip(ax.flat, MODS):
    hits = np.where((y == MODS.index(m)) & (snr == max(SNRS)))[0]
    if len(hits) == 0:
        continue
    i = hits[0]
    a.axis("on")
    a.plot(X[i, 0, :256], lw=.9, label="I"); a.plot(X[i, 1, :256], lw=.9, label="Q")
    a.set_title(m); a.set_xticks([]); a.set_yticks([])
ax.flat[0].legend(fontsize=7)
plt.suptitle(f"RadioML {DATASET}: one example per modulation at {max(SNRS)} dB (first {min(L, 256)} samples)")
plt.tight_layout(); plt.show()

## 2. Train the modulation classifier
- **features:** cumulants, spectral lines of z, z², z⁴, amplitude and frequency statistics, fed to gradient boosting
- **cnn:** 1-D CNN on raw IQ, trained with random carrier-phase rotation

Models are saved to `models/` so the agent (and `run_demo.py`) can reuse them.

In [ ]:
step(f"STEP 2  Train modulation classifier ({MODEL})")
results, classifiers = {}, {}

if MODEL in ("features", "both"):
    info("[features + GB] training")
    t0 = time.time()
    clf = ModulationClassifier("features", train_features_model(X[idx_tr], y[idx_tr], SEED, verbose=True), MODS, L)
    clf.save(f"models/classifier_features{TAG}.joblib")
    info(f"[features + GB] saved -> models/classifier_features{TAG}.joblib")
    info(f"[features + GB] predicting {len(idx_te)} test examples...")
    classifiers["features + GB"] = clf
    results["features + GB"] = clf.predict(X[idx_te], verbose=True)
    info(f"[features + GB] done in {time.time()-t0:.0f} s")

if MODEL in ("cnn", "both"):
    info(f"[1-D CNN] training for {EPOCHS} epochs (progress printed every epoch)")
    t0 = time.time()
    from src.classifier import train_cnn
    a, b = split(y[idx_tr], snr[idx_tr], test_size=0.1, seed=SEED)
    idx_fit, idx_val = idx_tr[a], idx_tr[b]
    net = train_cnn(X[idx_fit], y[idx_fit], X[idx_val], y[idx_val], len(MODS),
                    epochs=EPOCHS, save_path=f"models/_cnn_best{TAG}.pt", seed=SEED)
    clf = ModulationClassifier("cnn", net, MODS, L)
    clf.save(f"models/classifier_cnn{TAG}.pt")
    info(f"[1-D CNN] saved -> models/classifier_cnn{TAG}.pt")
    classifiers["1-D CNN"] = clf
    results["1-D CNN"] = clf.predict(X[idx_te])
    info(f"[1-D CNN] done in {time.time()-t0:.0f} s")

### Classifier results

In [ ]:
step("STEP 2b  Evaluate classifiers")
y_te, snr_te = y[idx_te], snr[idx_te]
rows = [dict(model=n, snr_db=s, accuracy=float(np.mean(p[snr_te == s] == y_te[snr_te == s])))
        for n, p in results.items() for s in SNRS]
metrics = pd.DataFrame(rows)
metrics.to_csv(f"results/classifier_metrics{TAG}.csv", index=False)

summary_clf = {}
for n, p in results.items():
    hi = snr_te >= 10
    summary_clf[n] = (np.mean(p == y_te), np.mean(p[hi] == y_te[hi]))
    info(f"{n:15s} overall accuracy {summary_clf[n][0]:.3f} | at SNR >= 10 dB: {summary_clf[n][1]:.3f}")
    per_mod = {m: np.mean(p[hi & (y_te == i)] == i) for i, m in enumerate(MODS)}
    info(f"{n:15s} hardest classes at SNR >= 10 dB: " +
         ", ".join(f"{m} {a:.2f}" for m, a in sorted(per_mod.items(), key=lambda kv: kv[1])[:3]))
info(f"saved results/classifier_metrics{TAG}.csv, plotting...")

plt.figure(figsize=(7, 4.5))
for n, g in metrics.groupby("model"):
    plt.plot(g.snr_db, 100 * g.accuracy, "o-", label=n)
plt.axhline(100 / len(MODS), color="gray", ls=":", label="random guess")
plt.xlabel("SNR [dB]"); plt.ylabel("test accuracy [%]"); plt.ylim(0, 100)
plt.title(f"RadioML {DATASET} modulation classification"); plt.grid(alpha=.3); plt.legend()
plt.tight_layout(); plt.savefig(f"results/classifier_acc_vs_snr{TAG}.png", dpi=130); plt.show()

sz = 7 if len(MODS) <= 12 else 11
fig, ax = plt.subplots(1, len(results), figsize=(sz * len(results), sz - 1), squeeze=False)
hi = snr_te >= 10
for a, (n, p) in zip(ax[0], results.items()):
    cm = confusion_matrix(y_te[hi], p[hi], labels=range(len(MODS)), normalize="true")
    a.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    a.set_xticks(range(len(MODS)), MODS, rotation=60); a.set_yticks(range(len(MODS)), MODS)
    for i in range(len(MODS)):
        for j in range(len(MODS)):
            if cm[i, j] > 0.05:
                a.text(j, i, f"{cm[i,j]:.2f}", ha="center", va="center", fontsize=7,
                       color="white" if cm[i, j] > 0.5 else "black")
    a.set(title=f"{n} (SNR ≥ 10 dB)", xlabel="predicted", ylabel="true")
plt.tight_layout(); plt.savefig(f"results/classifier_confusion{TAG}.png", dpi=130); plt.show()

Accuracy stays below 100% even at high SNR for known reasons. In 2016: QAM16 vs QAM64 (128 samples hold only about
16 symbols) and AM-DSB vs WBFM (many examples come from silent audio). In 2018: the high-order QAM and APSK classes
(e.g. 128QAM vs 256QAM) are the usual confusions.

In [ ]:
# the agent uses the most accurate model at high SNR
best_name = max(summary_clf, key=lambda n: summary_clf[n][1])
CLF = classifiers[best_name]
info(f"agent will use: {best_name}")

## 3. Simulate a deep-space pass
RadioML has no bits or telemetry, so we generate CCSDS-style telemetry frames (sync marker, randomizer, CRC-16) and
send them through a channel with the impairments the dataset README suggests adding:

| Impairment | Value in this pass |
|---|---|
| Doppler offset + linear drift | 0.05 cycles/sample, drifting |
| Frequency hop | +0.03 cycles/sample at about 68% of the pass |
| Dropout | 25,000 samples at about 35% of the pass |
| Phase noise, clock drift, silence before/after | yes |
| Noise | Es/N0 = `ESN0_DB` |

The agent is told **none** of these values.

In [ ]:
def simulate_pass(esn0, n_frames):
    tx, payloads, _ = transmit(n_frames, sps=SPS)
    n = len(tx) + 20000
    return deep_space_channel(tx, SPS, esn0_db=esn0, f0=0.05, doppler_rate=2e-8,
                              hop_at=int(0.68 * n), hop_df=0.03,
                              dropouts=[(int(0.35 * n), 25000)])

step("STEP 3  Simulate a deep-space pass")
info(f"generating {N_FRAMES} telemetry frames and sending them through the channel...")
t0 = time.time()
rx, truth = simulate_pass(ESN0_DB, N_FRAMES)
info(f"received {len(rx)} complex samples in {time.time()-t0:.1f} s")
info(f"hidden from the agent: Doppler {truth['f'][truth['lead_in']]:+.4f} cyc/sample, "
     f"hop at sample {truth['hop_at']}, dropout at {truth['dropouts']}, Es/N0 {ESN0_DB} dB")

## 4. Run the autonomous receiver agent
Find signal → estimate Doppler and drift (detect hops) → **identify modulation with the trained classifier** →
matched filter → timing recovery → carrier loop → frame sync → CRC → telemetry. After a dropout or hop it re-acquires on its own.

In [ ]:
step("STEP 4  Autonomous receiver agent")
info("agent starts with no knowledge of the signal; its own log follows")
t0 = time.time()
agent = SignalAgent(sps=SPS, classifier=CLF, verbose=True)
recs = agent.run(rx)
info(f"agent finished in {time.time()-t0:.1f} s: {len(agent.events)} events, {len(recs)} frames found")

In [ ]:
# the agent's decision log
events = pd.DataFrame(agent.events, columns=["sample", "event"])
events.to_csv("results/agent_events.csv", index=False)
info("saved results/agent_events.csv")
events

## 5. Output: decoded telemetry

In [ ]:
step("STEP 5  Decoded telemetry")
log = pd.DataFrame(recs)
log.to_csv("results/telemetry_log.csv", index=False)
recovered = len(set(log[log.crc_ok].seq) & set(range(N_FRAMES))) / N_FRAMES
info(f"frames found: {len(log)} | CRC ok: {log.crc_ok.sum()} | CRC failed: {(~log.crc_ok).sum()}")
info(f"unique frames recovered: {100*recovered:.1f}% of {N_FRAMES}")
missing = sorted(set(range(N_FRAMES)) - set(log[log.crc_ok].seq))
info(f"missing frames: {len(missing)} (first few: {missing[:10]})")
for seg, g in log.groupby("segment"):
    info(f"segment {seg}: {g.crc_ok.sum()} good frames, seq {g.seq.min()}..{g.seq.max()}")
info("saved results/telemetry_log.csv")
log[log.crc_ok].head(10)

In [ ]:
ok = log[log.crc_ok]
fig, ax = plt.subplots(1, 3, figsize=(14, 3.2))
for a, col, unit in zip(ax, ["temp_C", "bus_V", "current_A"], ["°C", "V", "A"]):
    a.plot(ok.time_ms / 1000, ok[col], ".", ms=3)
    a.set(title=col, xlabel="mission time [s]", ylabel=unit); a.grid(alpha=.3)
plt.suptitle("Decoded telemetry (gaps = dropout / hop)"); plt.tight_layout()
plt.savefig("results/telemetry_decoded.png", dpi=130); plt.show()
info("saved results/telemetry_decoded.png")

## 6. Before / after plots

In [ ]:
step("STEP 6  Before / after plots")
fig, ax = plt.subplots(2, 2, figsize=(13, 9))

f_b, p_b = welch(rx, nperseg=2048, return_onesided=False)
f_a, p_a = welch(agent.diag["filtered"][0], nperseg=2048, return_onesided=False)
for fr, pp, lab in [(f_b, p_b, "before: raw received IQ"),
                    (f_a, p_a, "after: agent's Doppler removal + matched filter")]:
    o = np.argsort(fr)
    ax[0, 0].plot(fr[o], 10*np.log10(pp[o] / pp.max()), label=lab, alpha=.8)
ax[0, 0].set(title="Spectrum before / after", xlabel="frequency [cycles/sample]", ylabel="PSD [dB, normalized]")
ax[0, 0].legend(); ax[0, 0].grid(alpha=.3)

ax[0, 1].specgram(rx, NFFT=1024, noverlap=512, Fs=1, cmap="viridis")
for (cb, fb, k, fit) in agent.diag["doppler"]:
    ax[0, 1].plot(k[::200], fit[::200], "r-", lw=1.5)
ax[0, 1].set(title="Spectrogram + agent's Doppler track (red)", xlabel="sample", ylabel="cycles/sample")

s0 = truth["lead_in"]
naive = rx[s0:s0 + 4000 * SPS:SPS]
sym = agent.diag["symbols"][0][500:4500]
ax[1, 0].plot(naive.real, naive.imag, ".", ms=2, alpha=.4, label="before (raw samples)")
ax[1, 0].plot(sym.real, sym.imag, ".", ms=2, alpha=.4, label="after (agent output)")
ax[1, 0].set(title="Constellation", aspect="equal", xlim=(-3, 3), ylim=(-3, 3)); ax[1, 0].legend()

ax[1, 1].plot(np.arange(len(agent.diag["block_snr"])) * agent.block, 10*np.log10(agent.diag["block_snr"]), lw=.8)
ax[1, 1].axhline(10*np.log10(agent.snr_gate), color="r", ls="--", label="acquisition gate")
for smp, msg in agent.events:
    ax[1, 1].axvline(smp, color="k", alpha=.25)
ax[1, 1].set(title="Blind SNR + agent events (grey lines)", xlabel="sample", ylabel="dB"); ax[1, 1].legend()
plt.tight_layout(); plt.savefig("results/overview.png", dpi=130); plt.show()
info("saved results/overview.png")

## 7. Performance vs SNR

In [ ]:
step("STEP 7  Performance vs SNR")
if not RUN_SWEEP:
    info("RUN_SWEEP = False, skipped")
else:
    snrs = np.arange(0, 13, 1.0)
    rates = []
    for i, e in enumerate(snrs):
        t0 = time.time()
        r, _ = simulate_pass(e, 120)
        out = SignalAgent(sps=SPS, verbose=False).run(r)
        rates.append(len({o["seq"] for o in out if o["crc_ok"]} & set(range(120))) / 120)
        info(f"[{i+1:2d}/{len(snrs)}] Es/N0 {e:4.1f} dB -> {100*rates[-1]:5.1f}% frames decoded ({time.time()-t0:.1f} s)")
    plt.figure(figsize=(6, 4))
    plt.plot(snrs, 100 * np.array(rates), "o-")
    plt.xlabel("Es/N0 [dB]"); plt.ylabel("frames decoded (CRC ok) [%]")
    plt.title("Frame success vs SNR (Doppler + drift + hop + dropout)\nceiling ≈ 83%: the dropout removes ~17% of frames")
    plt.grid(alpha=.3); plt.tight_layout(); plt.savefig("results/sweep.png", dpi=130); plt.show()
    info("saved results/sweep.png")

## 8. Summary (copy into your README)

In [ ]:
step("STEP 8  Summary")
mods_found = [m[1] for m in agent.diag.get("modulation", [])]
summary = pd.DataFrame([
    ("Dataset (training)", f"RadioML {DATASET}" + (f" (subset: {PER_PAIR} per modulation and SNR)" if DATASET == "2018" else "")),
    ("Training examples / window length", f"{len(idx_tr)} / {L} samples"),
    ("Classifier used by agent", best_name),
    ("Classifier accuracy, all SNRs", f"{summary_clf[best_name][0]:.3f}"),
    ("Classifier accuracy, SNR >= 10 dB", f"{summary_clf[best_name][1]:.3f}"),
    ("Simulated pass Es/N0", f"{ESN0_DB} dB"),
    ("Modulation identified by agent", ", ".join(mods_found) + "  (true: BPSK)"),
    ("Signal segments acquired", str(len(set(log.segment)))),
    ("Telemetry frames recovered (CRC ok)", f"{100*recovered:.1f}% of {N_FRAMES}"),
], columns=["metric", "value"])
summary.to_csv(f"results/summary{TAG}.csv", index=False)
info(f"saved results/summary{TAG}.csv")
info(f"ALL DONE in {time.time() - T_START:.0f} s. Outputs: " + ", ".join(sorted(os.listdir("results"))))
summary

## Limitations and next steps
- RadioML has no bit-level ground truth, so decoding is shown on synthetic CCSDS-style telemetry. Next: one real SatNOGS IQ recording.
- Only a BPSK demodulator exists. The classifier already tells the agent when it sees something else; next is adding QPSK and switching automatically.
- No forward error correction yet. A convolutional code with Viterbi decoding would extend decoding below about 4 dB.
- Domain gap: a classifier trained only on RadioML may not transfer perfectly to other channels. RadioML also has known errata (DeepSig).

**Credits:** RadioML: DeepSig Inc., CC BY-NC-SA 4.0 (O'Shea & West, GRCon 2016). SatNOGS: Libre Space Foundation.